# Example-13: Computation of mapping derivatives

In [1]:
# Import

import torch

import sys
sys.path.append('..')

from harmonica.derivative import derivative, evaluate
from harmonica.derivative import monomial_index, monomial_table, evaluate_table

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())

True


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# In this example computation of derivatives is shown
# Given a mapping(state, *args) -> result, derivatives of result with respect to state are computed
# Derivatives are computed by nesting jacobian, e.g. torch.func.jacfwd or torch.func.jacrev
# Note, such nesting is not efficient, in particular functorch.jacrev can be very memory expensive

In [4]:
# Define test mapping

# Here 'state' argument is a 'deviation' vector
# It contains deviations for quadrupole strength and length, and initial value of coordinate and momentum
# While 'knobs' argument contains 'unperturbed' values of quadrupole strength and length
# This function returns coordinate and momentum at the quadrupole exit

# Note, derivatives are computed with respect to the first argument
# Note, mapping can return arbitrary shape tensor

def mapping(state:torch.Tensor, knobs:torch.Tensor) -> torch.Tensor:
    k, l, q, p = state
    K, L = knobs
    Q = q*torch.cos(torch.sqrt(K + k)*(L + l)) + p*torch.sin(torch.sqrt(K + k)*(L + l))/torch.sqrt(K + k)
    P = p*torch.cos(torch.sqrt(K + k)*(L + l)) - q*torch.sin(torch.sqrt(K + k)*(L + l))*torch.sqrt(K + k)
    return torch.stack([Q, P])

In [5]:
# Evaluate test mapping for given deviation vector

state = torch.tensor([0.1, 0.1, 0.001, 0.001], dtype=dtype, device=device)
knobs = torch.tensor([1.0, 0.5], dtype=dtype, device=device)
value = mapping(state, knobs)
print(value)

tensor([1.369625086187e-03, 1.911539578908e-04], dtype=torch.float64)


In [6]:
# In this example deviation monomial is k^a l^b q^c p^d
# Given total monomial degree (a + b + c + d), all possible monomial exponents can be computed:

degree = 1
print(monomial_index(4, degree))
print()

degree = 2
print(monomial_index(4, degree))
print()

tensor([[1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1]])

tensor([[2, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 0, 1, 0],
        [1, 0, 0, 1],
        [1, 1, 0, 0],
        [0, 2, 0, 0],
        [0, 1, 1, 0],
        [0, 1, 0, 1],
        [1, 0, 1, 0],
        [0, 1, 1, 0],
        [0, 0, 2, 0],
        [0, 0, 1, 1],
        [1, 0, 0, 1],
        [0, 1, 0, 1],
        [0, 0, 1, 1],
        [0, 0, 0, 2]])



In [7]:
# Compute Taylor expansion of mapping at given point

# Set point and other arguments

state = torch.tensor([0.0, 0.0, 0.0, 0.0], dtype=dtype, device=device)
knobs = torch.tensor([1.0, 0.5], dtype=dtype, device=device)

# Set computation degree

degree = 1

# Compute derivatives

table = derivative(degree, mapping, state, knobs, jacobian=torch.func.jacfwd)

# The result is a tuple of derivative tensors (value, jacobian, hessian, ...)

for tensor in table:
    print(tensor.shape)
print()

# Monomial representation can be obtained with monomial_table

table = monomial_table(len(state), degree, table)

for key, value in table.items():
    print(f'{key}: {value.tolist()}')
print()

torch.Size([2])
torch.Size([2, 4])

(0, 0, 0, 0): [0.0, 0.0]
(1, 0, 0, 0): [0.0, 0.0]
(0, 1, 0, 0): [0.0, 0.0]
(0, 0, 1, 0): [0.8775825618903728, -0.479425538604203]
(0, 0, 0, 1): [0.479425538604203, 0.8775825618903728]



In [8]:
%%time

# Mapping value at different deviation state can be computed with evaluate (or evaluate_table)

# Compute derivatives
# Change degree value to observe convergence

degree = 4

state = torch.tensor([0.0, 0.0, 0.0, 0.0], dtype=dtype, device=device)
knobs = torch.tensor([1.0, 0.5], dtype=dtype, device=device)
table = derivative(degree, mapping, state, knobs)

# Evaluate at different state

state = torch.tensor([0.1, 0.1, 0.001, 0.001], dtype=dtype, device=device)
knobs = torch.tensor([1.0, 0.5], dtype=dtype, device=device)
value = mapping(state, knobs)

# Evaluate derivative representation

result = evaluate(state, table)
print(value - result)

# Monomian representation

table = monomial_table(len(state), degree, table)

# Evaluate monomial representation

result = evaluate_table(state, table)
print(value - result)

tensor([1.756743293088e-08, 7.367097959663e-08], dtype=torch.float64)
tensor([1.756743293088e-08, 7.367097959660e-08], dtype=torch.float64)
CPU times: user 54.9 ms, sys: 275 µs, total: 55.2 ms
Wall time: 55.4 ms


In [9]:
# Print monomial representation

for key, value in table.items():
    print(f'{key}: {value.tolist()}')
print()

(0, 0, 0, 0): [0.0, 0.0]
(1, 0, 0, 0): [0.0, 0.0]
(0, 1, 0, 0): [0.0, 0.0]
(0, 0, 1, 0): [0.8775825618903728, -0.479425538604203]
(0, 0, 0, 1): [0.479425538604203, 0.8775825618903728]
(2, 0, 0, 0): [0.0, 0.0]
(1, 1, 0, 0): [0.0, -0.0]
(1, 0, 1, 0): [-0.11985638465105075, -0.4591084097746947]
(1, 0, 0, 1): [-0.020317128829508313, -0.11985638465105075]
(0, 2, 0, 0): [0.0, -0.0]
(0, 1, 1, 0): [-0.479425538604203, -0.8775825618903728]
(0, 1, 0, 1): [0.8775825618903728, -0.479425538604203]
(0, 0, 2, 0): [0.0, -0.0]
(0, 0, 1, 1): [0.0, -0.0]
(0, 0, 0, 2): [0.0, -0.0]
(3, 0, 0, 0): [0.0, 0.0]
(2, 1, 0, 0): [0.0, 0.0]
(2, 0, 1, 0): [0.002539641103688539, 0.020061330288758422]
(2, 0, 0, 1): [0.00025579854074989083, 0.002539641103688539]
(1, 2, 0, 0): [0.0, -0.0]
(1, 1, 1, 0): [-0.4591084097746947, -0.757726177239322]
(1, 1, 0, 1): [-0.11985638465105074, -0.4591084097746947]
(1, 0, 2, 0): [0.0, -0.0]
(1, 0, 1, 1): [0.0, -0.0]
(1, 0, 0, 2): [0.0, -0.0]
(0, 3, 0, 0): [0.0, 0.0]
(0, 2, 1, 0): [-0.4